In [ ]:
# ============================================
# FEATURE ENGINEERING AND FEATURE SELECTION
# ============================================

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectFromModel

# Load dataset
df = pd.read_csv("/content/aml_data/SAML-D.csv")

print("Original dataset shape:", df.shape)


# ============================================
# 1. FEATURE ENGINEERING
# ============================================

# Convert Date column
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

# Date-based features
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Day"] = df["Date"].dt.day
df["DayOfWeek"] = df["Date"].dt.dayofweek

# Weekend indicator
df["Is_weekend"] = (df["DayOfWeek"] >= 5).astype(int)

# Extract hour from Time
df["Hour"] = pd.to_datetime(
    df["Time"],
    errors="coerce"
).dt.hour

# Night-time indicator
df["Is_night"] = (
    (df["Hour"] < 6) |
    (df["Hour"] >= 22)
).astype(int)

# Same currency indicator
df["Same_currency"] = (
    df["Payment Currency"] ==
    df["Received Currency"]
).astype(int)

# Cross-border transaction indicator
df["Cross_border"] = (
    df["Sender Bank Location"] !=
    df["Receiver Bank Location"]
).astype(int)

# Log transformation of transaction amount
df["Log_amount"] = np.log1p(df["Amount"])

print("\nFeature engineering completed.")
print("New dataset shape:", df.shape)


# ============================================
# 2. DEFINE FEATURES AND TARGET
# ============================================

X = df.drop("Is_laundering", axis=1)
y = df["Is_laundering"]

print("\nFeatures shape:", X.shape)
print("Target shape:", y.shape)


# ============================================
# 3. IDENTIFY NUMERICAL AND CATEGORICAL FEATURES
# ============================================

numerical_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

print("\nNumerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)


# ============================================
# 4. TRAIN-TEST SPLIT
# ============================================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("\nTraining set:", X_train.shape)
print("Testing set:", X_test.shape)


# ============================================
# 5. PREPROCESSING
# ============================================

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            ),
            categorical_features
        )
    ]
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("\nProcessed training shape:",
      X_train_processed.shape)

print("Processed testing shape:",
      X_test_processed.shape)


# ============================================
# 6. FEATURE SELECTION USING L1 LOGISTIC REGRESSION
# ============================================

selector_model = LogisticRegression(
    penalty="l1",
    solver="liblinear",
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)

selector_model.fit(
    X_train_processed,
    y_train
)


# ============================================
# 7. SELECT IMPORTANT FEATURES
# ============================================

selector = SelectFromModel(
    selector_model,
    threshold="median",
    prefit=True
)

X_train_selected = selector.transform(
    X_train_processed
)

X_test_selected = selector.transform(
    X_test_processed
)

print("\n============================================")
print("FEATURE SELECTION RESULTS")
print("============================================")

print("Features before selection:",
      X_train_processed.shape[1])

print("Features after selection:",
      X_train_selected.shape[1])

print("Features removed:",
      X_train_processed.shape[1] -
      X_train_selected.shape[1])


# ============================================
# 8. SUMMARY
# ============================================

print("\n============================================")
print("SUMMARY")
print("============================================")
print("Feature engineering completed")
print("Categorical features encoded")
print("Numerical features scaled")
print("L1 Logistic Regression used for feature selection")
print("SelectFromModel used with median threshold")
print("Selected features are ready for model training")